# G1 중복·분석단위·분할정책 결정지원 EDA — v001

**NOT A PRIMARY RESULT**  
**REQUI DISPOSITION**  

이 노트북은 Research Director가 정책을 선택하기 전에 관찰 구조와 선택지별 trade-off를 확인하도록 만든 **탐색적 결정지원 자료**입니다. exact duplicate 처리, 기술통계 분모, source/domain prevalence 분모, predictive holdout grouping, cross-direction sensitivity, 500건 manual-QC 배분을 이 노트북이 확정하지 않습니다.

공개 출력에는 한국어·영어 원문 열을 포함하지 않습니다. tokenization, morphology, TP 추정, QC disposition, 정규화, 모델 적합을 실행하지 않습니다.

## 먼저 구분할 세 가지 identity

| 구분 | 필드 | 이 노트북에서의 의미 |
|---|---|---|
| 관측 행 identity | `pair_id` | provenance에 연결된 registry 한 행 |
| exact-content identity | `duplicate_group_id` | 내용이 정확히 같은 행들의 그룹 |
| 결정론적 pointer | `representative_pair_id` | 그룹을 대표하는 provenance pointer |

세 필드는 서로 바꾸어 쓸 수 없습니다. 특히 representative의 metadata를 그룹 전체의 의미 covariate로 자동 승격하지 않습니다.

## 분석 흐름

`canonical registry (read-only)` → `identity map` → `duplicate composition` → `cross-direction` → `split leakage` → `identifiability` → `scenario trade-off` → `manual-QC allocation` → `Director decision requests`

각 시각화는 하나의 질문만 답하도록 구성합니다. 비율은 표와 manifest에서 분모를 함께 기록합니다.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[2]

REPORT_DIR = PROJECT_ROOT / 'outputs/reports/eda_g1_decision_support'
FIGURE_DIR = PROJECT_ROOT / 'outputs/figures/eda_g1_decision_support'
MANIFEST_DIR = PROJECT_ROOT / 'outputs/manifests/eda_g1_decision_support'
SCRIPT = PROJECT_ROOT / 'notebooks/exploratory/eda/build_eda_g1_duplicate_decision_support_v001.py'

pair_candidates = [
    PROJECT_ROOT / 'data/registry/PAIR_REGISTRY_v001.parquet',
    PROJECT_ROOT.parent / 'codex-g1/data/registry/PAIR_REGISTRY_v001.parquet',
]
source_candidates = [
    PROJECT_ROOT / 'data/registry/SOURCE_REGISTRY_v001.parquet',
    PROJECT_ROOT.parent / 'codex-g1/data/registry/SOURCE_REGISTRY_v001.parquet',
]
PAIR_REGISTRY = next((path for path in pair_candidates if path.exists()), None)
SOURCE_REGISTRY = next((path for path in source_candidates if path.exists()), None)
INPUT_MANIFEST = PROJECT_ROOT / 'outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json'
assert PAIR_REGISTRY is not None and SOURCE_REGISTRY is not None and INPUT_MANIFEST.exists(), 'canonical registry/manifest 경로를 확인하세요.'
print('프로젝트:', PROJECT_ROOT)
print('pair registry:', PAIR_REGISTRY)
print('source registry:', SOURCE_REGISTRY)

### 실행 안내

다음 셀은 기본적으로 전체 EDA 엔진을 실행합니다. 565만 행을 직접 DataFrame으로 올리지 않고 DuckDB aggregation과 `/tmp` spill을 사용합니다. 이미 검증된 산출물을 다시 표시할 때는 환경변수 `EDA_G1_REBUILD=0`으로 실행할 수 있습니다.

In [ ]:
REBUILD = os.environ.get('EDA_G1_REBUILD', '1') == '1'
if REBUILD:
    command = [
        sys.executable, str(SCRIPT),
        '--pair-registry', str(PAIR_REGISTRY),
        '--source-registry', str(SOURCE_REGISTRY),
        '--input-manifest', str(INPUT_MANIFEST),
        '--memory-limit', '6GB',
        '--threads', '4',
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('기존 검증 산출물을 사용합니다: EDA_G1_REBUILD=0')

In [ ]:
summary = pd.read_csv(REPORT_DIR / 'EDA_G1_DUPLICATE_DECISION_SUPPORT_v001.csv')
stratum = pd.read_csv(REPORT_DIR / 'EDA_G1_DUPLICATE_STRATUM_PROFILE_v001.csv')
split_matrix = pd.read_csv(REPORT_DIR / 'EDA_G1_SPLIT_LEAKAGE_MATRIX_v001.csv')
cross_direction = pd.read_csv(REPORT_DIR / 'EDA_G1_CROSS_DIRECTION_PROFILE_v001.csv')
manual_frame = pd.read_csv(REPORT_DIR / 'EDA_G1_MANUAL_QC_SAMPLING_FRAME_v001.csv')
ident_diag = pd.read_csv(REPORT_DIR / 'EDA_G1_IDENTIFIABILITY_DIAGNOSTICS_v001.csv')
manifest = json.loads((MANIFEST_DIR / 'EDA_G1_DECISION_SUPPORT_MANIFEST_v001.json').read_text(encoding='utf-8'))
print('검증 상태:', manifest['validation_status'])
print('입력 행:', f"{manifest['identity_counts']['row_count']:,}")
print('exact groups:', f"{manifest['identity_counts']['group_count']:,}")

## A. Registry와 identity map

**질문:** exact duplicate collapse를 하면 registry 규모가 얼마나 바뀌는가?

- 관찰 단위: 행, `pair_id`, duplicate group, representative row를 분리합니다.
- 제한: 크기 분포는 exact-content identity만 설명하며 의미 품질을 평가하지 않습니다.
- 결정 질문: primary estimate의 단위를 행으로 둘지, group으로 둘지, cluster-aware inference를 사용할지 Director가 선택해야 합니다.

In [ ]:
identity_view = summary[summary['scenario'].eq('IDENTITY')][
    ['metric', 'numerator', 'denominator', 'rate', 'stratum', 'interpretation_boundary']
].copy()
display(identity_view)
display(Image(filename=str(FIGURE_DIR / 'F01_DUPLICATE_GROUP_SIZE_DISTRIBUTION_v001.png'), width=1200))

## B. Corpus/source별 중복 집중도와 분모 민감도

**질문:** 모든 행을 분모로 쓸 때와 representative 행을 분모로 쓸 때 어떤 구성비가 달라지는가?

아래 표의 `row_share`는 registry 행 분모, `representative_share`는 representative 행 분모입니다. `group_presence_share`는 한 그룹이 여러 strata에 걸치면 각 stratum에 나타날 수 있으므로 합이 항상 1이 되는 prevalence 분모가 아닙니다.

In [ ]:
corpus_burden = stratum[stratum['axis'].eq('logical_corpus')][
    ['stratum_value', 'row_count', 'duplicate_group_presence_count',
     'rows_in_non_singleton_groups', 'duplicate_affected_row_rate',
     'representative_row_count', 'row_share', 'representative_share',
     'composition_share_change']
]
display(corpus_burden)
display(Image(filename=str(FIGURE_DIR / 'F02_DUPLICATE_BURDEN_BY_CORPUS_SOURCE_v001.png'), width=1250))

## C. Cross-direction duplicate 구조

**질문:** exact group 안의 raw direction은 하나인가, UNKNOWN뿐인가, 여러 known direction이 섞였는가?

분류는 `translation_direction_raw`를 사용합니다. group-resolved canonical direction이 `UNKNOWN`인 경우에도 raw provenance의 혼합을 지우지 않습니다. 원인을 추정하거나 새 direction-resolution rule을 만들지 않습니다.

In [ ]:
direction_summary = (
    cross_direction[cross_direction['axis'].eq('group_size_bucket')]
    .groupby('direction_composition_class', as_index=False)
    .agg(duplicate_group_count=('duplicate_group_count', 'sum'),
         affected_row_count=('affected_row_count', 'sum'))
)
display(direction_summary)
display(Image(filename=str(FIGURE_DIR / 'F03_CROSS_DIRECTION_COMPOSITION_v001.png'), width=1250))

## D. Upstream split leakage

**질문:** source가 제공한 train/validation label을 project holdout으로 그대로 재사용할 수 있는가?

이 label은 provenance입니다. matrix의 off-diagonal `TRAIN × VALIDATION` cell은 양쪽에 동시에 나타나는 exact-content group 수입니다. 이 분석은 future near-duplicate overlap을 측정하지 않습니다.

In [ ]:
display(split_matrix[split_matrix['record_type'].eq('SUMMARY')])
display(Image(filename=str(FIGURE_DIR / 'F04_UPSTREAM_SPLIT_LEAKAGE_MATRIX_v001.png'), width=900))
display(summary[summary['scenario'].str.contains('HASH_80_20', na=False)][
    ['scenario', 'metric', 'numerator', 'denominator', 'rate', 'stratum', 'interpretation_boundary']
])

## E. Source/domain/direction identifiability

**질문:** corpus 안에서 source, domain, direction을 서로 분리해 해석할 자료 구조가 있는가?

zero cell, constant factor, deterministic mapping을 확인합니다. 이는 모형의 계수표가 아니며 causal effect를 뜻하지 않습니다. 026의 raw source–domain correspondence는 실제 contingency table로 검증합니다.

In [ ]:
display(ident_diag[[
    'portfolio', 'crosstab', 'x_level_count', 'y_level_count',
    'possible_cell_count', 'observed_cell_count', 'zero_cell_rate',
    'x_deterministic_row_rate', 'y_deterministic_row_rate', 'evidence_label'
]])
display(Image(filename=str(FIGURE_DIR / 'F05_IDENTIFIABILITY_HEATMAPS_v001.png'), width=1300))

## F. 정책 시나리오별 구성 영향

**질문:** S0–S3를 적용한다고 가정하면 행 수와 corpus/source/domain/direction 구성은 어떻게 달라지는가?

시나리오는 trade-off를 보여주는 **POLICY OPTION**입니다. 어느 시나리오도 선택하지 않습니다. S1의 representative metadata는 provenance pointer이며 group conflict를 해소하지 않습니다.

In [ ]:
scenario_rows = summary[summary['metric'].eq('scenario_row_count')][
    ['scenario', 'numerator', 'denominator', 'rate', 'stratum', 'interpretation_boundary']
]
display(scenario_rows)
display(Image(filename=str(FIGURE_DIR / 'F06_SCENARIO_COMPOSITION_IMPACT_v001.png'), width=1250))

## G. 500건 manual-QC sampling frame

**질문:** proportional allocation과 risk-oversampled allocation이 어떤 strata를 얼마나 대표하는가?

실제 표본은 뽑지 않습니다. rare/high-risk cell에는 minimum-per-cell logic을 적용하고, risk oversampling 후 population description이 필요하면 `N/n` weight를 사용해야 합니다. 최종 배분과 estimator는 Director 승인 사항입니다.

In [ ]:
allocation_rollup = (
    manual_frame.groupby('structural_risk_class', as_index=False)
    .agg(population_row_count=('population_row_count', 'sum'),
         proportional_allocation_n=('proportional_allocation_n', 'sum'),
         risk_oversampled_allocation_n=('risk_oversampled_allocation_n', 'sum'))
)
display(allocation_rollup)
print('proportional 합계:', manual_frame['proportional_allocation_n'].sum())
print('risk-oversampled 합계:', manual_frame['risk_oversampled_allocation_n'].sum())
display(Image(filename=str(FIGURE_DIR / 'F07_MANUAL_QC_ALLOCATION_v001.png'), width=1100))

## H. Director dashboard와 미결정 요청

아래 dashboard는 결론이 아니라 선택이 필요한 구조를 한 화면에 모은 것입니다. 세부 선택지는 `DECISION_REQUESTS_v001.md`에서 evidence artifact와 직접 연결합니다.

In [ ]:
display(Image(filename=str(FIGURE_DIR / 'F08_DIRECTOR_DECISION_DASHBOARD_v001.png'), width=1100))
display(Markdown((REPORT_DIR / 'DECISION_REQUESTS_v001.md').read_text(encoding='utf-8')))

## 해석 scaffold

**관찰:** registry 행과 exact-content group의 규모, 중복 집중 strata, cross-direction·cross-split 구조, deterministic mapping을 기술했습니다.

**가능한 메커니즘:** 현재 EDA는 중복이 생긴 원인이나 translation provenance의 생성 과정을 확정하지 않습니다.

**제한:** exact duplicate까지만 측정했습니다. metadata length proxy와 near-duplicate cluster는 현재 canonical artifact에 없습니다. representative pointer는 group semantic covariate가 아닙니다.

**결정:** D1–D7은 Research Director의 명시적 disposition이 필요합니다.

**결론:** 이 노트북은 **NOT A PRIMARY RESULT**이며 정책 선택, Phase-2 QC, cohort freeze, tokenization, morphology, TP estimation, model fitting으로 진행하지 않습니다.

In [ ]:
manifest_view = {
    'analysis_base_commit': manifest['analysis_base_commit'],
    'execution_code_commit': manifest['execution_code_commit'],
    'artifact_record_commit': manifest['artifact_record_commit'],
    'pair_registry_sha256': manifest['input_artifacts'][0]['sha256'],
    'source_registry_sha256': manifest['input_artifacts'][1]['sha256'],
    'sample_seed': manifest['runtime_policy']['sample_seed'],
    'sample_draw': manifest['runtime_policy']['sample_draw'],
    'output_count': len(manifest['outputs']),
}
display(pd.Series(manifest_view, name='manifest'))